<a href="https://colab.research.google.com/github/fahadabdullah-lab/smap-dispatch-gee-south-texas/blob/main/notebooks/02_data_pull_daily_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Cell 0 — Install + imports**

In [ ]:
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

### **Cell 1 — Authenticate + initialize + Composite Window**

In [ ]:

# -----------------------------
# 1) Authenticate + Initialize (required per new runtime)
# -----------------------------
try:
    ee.Initialize(project="ee-fafahadabdullah")
    print("✅ Earth Engine initialized (project=ee-fafahadabdullah)")
except Exception:
    print("🔐 Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project="ee-fafahadabdullah")
    print("✅ Earth Engine initialized (project=ee-fafahadabdullah)")

# -----------------------------
# 2) Composite window settings
# -----------------------------
TEST_DATE = "2021-08-15"    # center date (YYYY-MM-DD)  <-- change if needed
COMPOSITE_DAYS = 10         # total window length
HALF_WINDOW = COMPOSITE_DAYS // 2

center = ee.Date(TEST_DATE)
start_date = center.advance(-HALF_WINDOW, "day")
end_date   = center.advance(HALF_WINDOW + 1, "day")  # end is exclusive (safer)

print("📅 Center date:", TEST_DATE)
print("🧱 Composite window days:", COMPOSITE_DAYS)
print("⏱️ Start:", start_date.format("YYYY-MM-dd").getInfo())
print("⏱️ End  :", end_date.format("YYYY-MM-dd").getInfo(), "(end exclusive)")

### **Cell 2 - AOI Drawing**

In [ ]:
import geemap

# ------------------------------------------------
# Interactive map for drawing AOI
# ------------------------------------------------

Map = geemap.Map(center=[26.1, -97.3], zoom=9)

Map.add_basemap("HYBRID")

# enable drawing tool
Map.add_draw_control()

print("🟨 Please draw your AOI polygon on the map using the drawing tool.")

Map

### **Cell 3 - Convert drawn polygon → AOI**



In [ ]:
# ------------------------------------------------
# Convert drawn polygon to Earth Engine geometry
# ------------------------------------------------

drawn_features = Map.draw_control.data

if drawn_features is None or len(drawn_features) == 0:
    raise ValueError("❌ No polygon detected. Please draw an AOI first.")

# Convert to Earth Engine FeatureCollection
drawn_fc = ee.FeatureCollection(drawn_features)

# Extract geometry
aoi = ee.Feature(drawn_fc.first()).geometry()

print("✅ AOI successfully converted to Earth Engine geometry")

# ------------------------------------------------
# Print AOI area
# ------------------------------------------------

area_km2 = aoi.area().divide(1e6)

print("AOI area (km²):", area_km2.getInfo())

# ------------------------------------------------
# Visualize AOI
# ------------------------------------------------

Map.addLayer(aoi, {"color": "yellow"}, "AOI Geometry")

Map

## **Cell 4 - Load SMAP and create 10-day composite**

In [ ]:
# ------------------------------------------------
# Load SMAP Soil Moisture (10-day composite)
# ------------------------------------------------

SMAP_COLLECTION = "NASA/SMAP/SPL3SMP_E/005"

smap_ic = (
    ee.ImageCollection(SMAP_COLLECTION)
    .filterDate(start_date, end_date)
    .select("soil_moisture_am")
)

print("SMAP images in window:", smap_ic.size().getInfo())

# ------------------------------------------------
# Create composite
# ------------------------------------------------

smap_composite = smap_ic.mean().clip(aoi)

print("✅ SMAP composite created")

# ------------------------------------------------
# Visualization
# ------------------------------------------------

smap_vis = {
    "min": 0.05,
    "max": 0.45,
    "palette": ["red", "yellow", "green"]
}

Map.addLayer(smap_composite, smap_vis, "SMAP Soil Moisture (10-day mean)")

Map

### **Cell 5 - Load MODIS LST (temperature axis of DISPATCH)**

In [ ]:
# ------------------------------------------------
# MODIS LST (10-day composite)
# ------------------------------------------------

MODIS_LST = "MODIS/061/MOD11A1"

lst_ic = (
    ee.ImageCollection(MODIS_LST)
    .filterDate(start_date, end_date)
    .select(["LST_Day_1km", "QC_Day"])
)

print("MODIS LST images in window:", lst_ic.size().getInfo())

# ------------------------------------------------
# Apply scale factor
# ------------------------------------------------

def scale_lst(image):

    lst = image.select("LST_Day_1km").multiply(0.02)

    # Kelvin → Celsius
    lst = lst.subtract(273.15)

    return lst.copyProperties(image, image.propertyNames())


lst_scaled = lst_ic.map(scale_lst)

# ------------------------------------------------
# Composite
# ------------------------------------------------

lst_composite = lst_scaled.mean().clip(aoi)

print("✅ MODIS LST composite created")

# ------------------------------------------------
# Visualization
# ------------------------------------------------

lst_vis = {
    "min": 20,
    "max": 45,
    "palette": ["blue", "cyan", "yellow", "red"]
}

Map.addLayer(lst_composite, lst_vis, "MODIS LST (10-day mean °C)")

Map

### **Cell 6 - MODIS LST composite with QC filtering**

In [ ]:
# ------------------------------------------------
# MODIS LST (QC-filtered 10-day composite)
# ------------------------------------------------

MODIS_LST = "MODIS/061/MOD11A1"

lst_ic = (
    ee.ImageCollection(MODIS_LST)
    .filterDate(start_date, end_date)
    .select(["LST_Day_1km", "QC_Day"])
)

print("MODIS LST images in window:", lst_ic.size().getInfo())

def lst_qc_scale(img):
    # QC band
    qc = img.select("QC_Day")

    # Common conservative QC:
    # bits 0-1: mandatory QA (0 = good)
    # bits 2-3: data quality   (0 = good)
    mandatory_qa = qc.bitwiseAnd(3)                  # 0..3
    data_quality = qc.rightShift(2).bitwiseAnd(3)    # 0..3
    good = mandatory_qa.eq(0).And(data_quality.eq(0))

    # Scale LST to Kelvin then to Celsius
    lst_c = img.select("LST_Day_1km").multiply(0.02).subtract(273.15)

    return (lst_c
            .updateMask(good)
            .copyProperties(img, img.propertyNames()))

lst_scaled_qc = lst_ic.map(lst_qc_scale)

# Composite (mean) and clip early to AOI
lst_composite_qc = lst_scaled_qc.mean().clip(aoi)

print("✅ QC-filtered MODIS LST composite created")

# Visualization
lst_vis = {
    "min": 20,
    "max": 45,
    "palette": ["blue", "cyan", "yellow", "red"]
}

Map.addLayer(lst_composite_qc, lst_vis, "MODIS LST (QC-filtered mean °C)")
Map

### **Cell 7 - Load MODIS Reflectance and compute NDVI**

In [ ]:
# ------------------------------------------------
# MODIS Surface Reflectance (for NDVI)
# ------------------------------------------------

MODIS_REFL = "MODIS/061/MOD09GA"

refl_ic = (
    ee.ImageCollection(MODIS_REFL)
    .filterDate(start_date, end_date)
    .select(["sur_refl_b01", "sur_refl_b02"])
)

print("MODIS Reflectance images:", refl_ic.size().getInfo())

# ------------------------------------------------
# Scale reflectance
# ------------------------------------------------

def scale_reflectance(img):

    red = img.select("sur_refl_b01").multiply(0.0001)
    nir = img.select("sur_refl_b02").multiply(0.0001)

    return ee.Image.cat([red.rename("RED"), nir.rename("NIR")])\
        .copyProperties(img, img.propertyNames())

refl_scaled = refl_ic.map(scale_reflectance)

# ------------------------------------------------
# Composite
# ------------------------------------------------

refl_composite = refl_scaled.mean().clip(aoi)

print("✅ MODIS reflectance composite created")

# ------------------------------------------------
# NDVI calculation
# ------------------------------------------------

ndvi = refl_composite.normalizedDifference(["NIR", "RED"]).rename("NDVI")

print("✅ NDVI computed")

# ------------------------------------------------
# Visualization
# ------------------------------------------------

ndvi_vis = {
    "min": 0,
    "max": 0.8,
    "palette": ["brown", "yellow", "green"]
}

Map.addLayer(ndvi, ndvi_vis, "MODIS NDVI (10-day composite)")

Map

### **Cell 8 - Build the DISPATCH feature stack (NDVI + LST)**

In [ ]:
# ------------------------------------------------
# Build MODIS stack (NDVI + LST)
# ------------------------------------------------

modis_stack = ee.Image.cat([
    ndvi.rename("NDVI"),
    lst_composite_qc.rename("LST")
]).clip(aoi)

print("✅ MODIS stack created")

print("Bands in stack:", modis_stack.bandNames().getInfo())

Map.addLayer(modis_stack.select("NDVI"), {"min":0,"max":0.8}, "NDVI (stack)")
Map.addLayer(modis_stack.select("LST"), {"min":20,"max":45}, "LST (stack)")

Map

### **Cell 9 - Safe sampling for triangle analysis**

Now we extract pixels for the NDVI–LST triangle. We use controlled sampling (~20k pixels).

In [ ]:
# ------------------------------------------------
# Sample MODIS pixels for triangle analysis
# ------------------------------------------------

sample_pixels = modis_stack.sample(
    region=aoi,
    scale=1000,
    numPixels=20000,
    seed=42,
    geometries=False
)

print("Sampling MODIS pixels...")

# Convert to dataframe
sample_df = geemap.ee_to_df(sample_pixels)

print("✅ Sample size:", len(sample_df))

sample_df.head()

### **Cell 10 - Plot the NDVI–LST triangle**

In [ ]:
import matplotlib.pyplot as plt

# ------------------------------------------------
# NDVI-LST triangle visualization
# ------------------------------------------------

plt.figure(figsize=(7,5))

plt.scatter(
    sample_df["NDVI"],
    sample_df["LST"],
    s=20,
    alpha=0.7
)

plt.xlabel("NDVI")
plt.ylabel("LST (°C)")
plt.title("NDVI–LST Triangle (Bahía Grande)")

plt.grid(True)
plt.show()

In [ ]:
sample_df.to_csv("dispatch_triangle_samples.csv", index=False)

print("Triangle dataset saved")